# Volume 2. FSM And PFA Utilities

Question: how do the automata helpers expose discrete computation over assembly-coded states?

The transition logic is explicit and inspectable. The neural substrate supplies state and symbol assemblies that can be plotted, read out, compared, and composed with other operations.


In [ ]:
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

from neural_assemblies.assembly_calculus import FSMNetwork, PFANetwork, chance_overlap
from neural_assemblies.core.brain import Brain
from neural_assemblies.viz import plot_assemblies, plot_overlap_matrix


A deterministic parity machine toggles on `1` and stays put on `0`. The table is the complete computation.


In [ ]:
fsm_brain = Brain(p=0.05, save_winners=True, seed=42, engine="numpy_sparse")
fsm = FSMNetwork(
    fsm_brain,
    states=["even", "odd"],
    symbols=["0", "1"],
    transitions=[
        ("even", "0", "even"),
        ("even", "1", "odd"),
        ("odd", "0", "odd"),
        ("odd", "1", "even"),
    ],
    initial_state="even",
    n=4_000,
    k=50,
    beta=0.08,
    rounds=6,
    prefix="parity",
)

pd.DataFrame([
    {"from_state": state, "symbol": symbol, "to_state": to_state}
    for (state, symbol), to_state in sorted(fsm.transition_table.items())
])


In [ ]:
fig, axes = plot_assemblies(
    [fsm.state_lexicon["even"], fsm.state_lexicon["odd"]],
    fsm.n,
    titles=["state: even", "state: odd"],
    subtitles=[f"chance overlap ~= {chance_overlap(fsm.k, fsm.n):.3f}" for _ in range(2)],
    colors=["#4d9de0", "#e15554"],
    marker_size=16,
)
plt.show()
plt.close(fig)


In [ ]:
ax, matrix = plot_overlap_matrix(
    [fsm.state_lexicon["even"], fsm.state_lexicon["odd"]],
    labels=["even", "odd"],
    cmap="viridis",
)
ax.set_title("FSM state assembly overlap")
plt.show()
plt.close(ax.figure)


Running an input string gives a state trajectory. The final state answers whether the number of `1` symbols was even.


In [ ]:
input_symbols = list("1011010")
fsm.reset()
trajectory = fsm.run(input_symbols)

pd.DataFrame({
    "step": range(1, len(input_symbols) + 1),
    "input": input_symbols,
    "state_after": trajectory,
    "accepts_even_ones": [state == "even" for state in trajectory],
})


A probabilistic finite automaton uses the same idea, but some transitions branch. The current helper supports this through a neural coin-flip area for competing target states.


In [ ]:
pfa_brain = Brain(p=0.05, save_winners=True, seed=99, engine="numpy_sparse")
pfa = PFANetwork(
    pfa_brain,
    states=["dry", "rain"],
    symbols=["day"],
    transitions=[
        ("dry", "day", "dry", 0.70),
        ("dry", "day", "rain", 0.30),
        ("rain", "day", "dry", 0.40),
        ("rain", "day", "rain", 0.60),
    ],
    initial_state="dry",
    n=3_000,
    k=40,
    beta=0.08,
    rounds=5,
    prefix="weather",
)

pfa_rows = []
for seed in range(8):
    pfa.reset()
    trajectory = pfa.run(["day"] * 5, seed=seed)
    pfa_rows.append({
        "seed": seed,
        "trajectory": " -> ".join(["dry"] + trajectory),
        "final_state": trajectory[-1],
    })
pfa_runs = pd.DataFrame(pfa_rows)
pfa_runs


In [ ]:
counts = Counter(pfa_runs["final_state"])
fig, ax = plt.subplots(figsize=(5.2, 3.2))
bars = ax.bar(counts.keys(), counts.values(), color=["#f2c14e", "#4d9de0"])
ax.bar_label(bars, padding=3)
ax.set_ylabel("runs ending in state")
ax.set_title("PFA sample outcomes after five days")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()
plt.close(fig)


The automata notebooks are intentionally honest about the boundary: transition choice is explicit and auditable, while state and symbol representations remain assembly-coded and available for neural-style composition.
